# Phase 2 · Notebook 8 — People at risk

A risk map is more actionable when it's expressed in **people**. Here we overlay the ML susceptibility surface with **WorldPop** population and aggregate to **district** level, producing a watch-list of how many residents live in high-susceptibility zones.

Outputs: `district_risk_scores.csv` and a choropleth map.

## Step 1 — Download population (WorldPop 2020)

WorldPop gives **people per ~100 m cell**. We pull it for the AOI from Earth Engine.

In [1]:
import ee, geemap
import numpy as np
import rioxarray as rxr
import geopandas as gpd
import osmnx as ox
import pandas as pd
import matplotlib.pyplot as plt
from rasterio.features import rasterize
from shapely.geometry import box
from pathlib import Path

ee.Initialize(project="urban-flood-analysis-ncr-in")
REPO = Path.cwd().parent if Path.cwd().name in ("phase2", "notebooks") else Path.cwd()
OUT = REPO / "data" / "outputs"; RAW = REPO / "data" / "raw"
W, S, E, N = 77.18, 28.50, 77.38, 28.78
aoi = ee.Geometry.Rectangle([W, S, E, N])

pop_img = (ee.ImageCollection("WorldPop/GP/100m/pop").filter(ee.Filter.eq("year", 2020))
           .filterBounds(aoi).mosaic().clip(aoi))
geemap.download_ee_image(pop_img, str(RAW / "worldpop.tif"), scale=100, region=aoi, crs="EPSG:4326")
print("WorldPop downloaded.")

/Users/user/miniconda3/envs/yamuna-flood/lib/python3.11/site-packages/geemap/common.py:12434: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


  0%|          |0/1 tiles [00:00<?]

WorldPop downloaded.


/Users/user/miniconda3/envs/yamuna-flood/lib/python3.11/site-packages/geedim/image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'None'.
  return STACClient().get(self.id)


## Step 2 — Align susceptibility to the population grid and count people at risk

We resample the ML susceptibility onto the population grid, then count people living where susceptibility is **≥ 0.5** (the High / Very-High zones).

In [2]:
pop = rxr.open_rasterio(RAW / "worldpop.tif").squeeze()
susc = rxr.open_rasterio(OUT / "flood_susceptibility_ml.tif").squeeze().rio.reproject_match(pop)

popv = np.nan_to_num(pop.values.astype("float64")); popv[popv < 0] = 0
suscv = np.nan_to_num(susc.values.astype("float64"))
at_risk = popv * (suscv >= 0.5)

print(f"Total population in AOI:        {popv.sum():,.0f}")
print(f"People in high-risk zones:      {at_risk.sum():,.0f}")
print(f"Share of population at risk:    {100 * at_risk.sum() / popv.sum():.1f}%")

Total population in AOI:        10,195,527
People in high-risk zones:      612,846
Share of population at risk:    6.0%


## Step 3 — Aggregate to districts

We pull district boundaries from OpenStreetMap (restricted to a clean, non-overlapping set of actual districts), paint them onto the population grid, and sum people-at-risk inside each.

In [3]:
DISTRICTS = ["Central Delhi", "East Delhi", "New Delhi", "North Delhi", "North East Delhi",
             "North West Delhi", "Shahdara", "South Delhi", "South East Delhi", "South West Delhi",
             "West Delhi", "Gautam Buddha Nagar", "Ghaziabad", "Faridabad"]

d = ox.features_from_bbox(bbox=(W, S, E, N), tags={"boundary": "administrative", "admin_level": "5"})
d = d[d.geom_type.isin(["Polygon", "MultiPolygon"])].to_crs("EPSG:4326").dropna(subset=["name"])
d = d[d["name"].isin(DISTRICTS)].drop_duplicates("name")
d = d[d.intersects(box(W, S, E, N))].reset_index(drop=True)

transform = pop.rio.transform()
ids = {n: i + 1 for i, n in enumerate(d["name"])}
drast = rasterize([(g, ids[n]) for g, n in zip(d.geometry, d["name"])],
                  out_shape=pop.shape, transform=transform, fill=0, dtype="int32")

rows = []
for n, i in ids.items():
    m = drast == i
    p, r = popv[m].sum(), at_risk[m].sum()
    ms = float(suscv[m].mean()) if m.any() else 0.0
    if p > 0:
        rows.append({"district": n, "population": int(p), "pop_at_risk": int(r),
                     "pct_at_risk": round(100 * r / p, 1), "mean_susceptibility": round(ms, 3)})

tbl = pd.DataFrame(rows).sort_values("pop_at_risk", ascending=False).reset_index(drop=True)
tbl.to_csv(OUT / "district_risk_scores.csv", index=False)
tbl

,district,population,pop_at_risk,pct_at_risk,mean_susceptibility
0,East Delhi,1198228,183609,15.3,0.289
1,Central Delhi,1305585,173388,13.3,0.304
2,North East Delhi,1379113,79962,5.8,0.262
3,North Delhi,307867,58733,19.1,0.261
4,South East Delhi,1352249,45581,3.4,0.079
5,Gautam Buddha Nagar,835442,31419,3.8,0.118
6,Ghaziabad,1584275,27158,1.7,0.148
7,Shahdara,1333785,9168,0.7,0.036
8,Faridabad,15389,3586,23.3,0.451
9,North West Delhi,24185,220,0.9,0.040


## Step 4 — Choropleth: people at risk by district

In [4]:
dd = d.merge(tbl, left_on="name", right_on="district", how="inner")
dd = gpd.clip(dd, box(W, S, E, N))

fig, ax = plt.subplots(figsize=(9, 11))
dd.plot(column="pop_at_risk", cmap="OrRd", legend=True, edgecolor="white", linewidth=0.8, ax=ax,
        legend_kwds={"label": "People in high-risk zones", "shrink": 0.5})
for _, r in dd.iterrows():
    c = r.geometry.representative_point()
    ax.annotate(f"{r['district']}\n{r['pop_at_risk']:,}", (c.x, c.y),
                ha="center", va="center", fontsize=7.5, color="#1a1a1a")
ax.set_title("People living in high flood-susceptibility zones\nby district · Yamuna corridor, Delhi",
             fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUT / "people_at_risk_by_district.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved district_risk_scores.csv and people_at_risk_by_district.png")

Saved district_risk_scores.csv and people_at_risk_by_district.png


/var/folders/jr/0tppzf_n6rscbz7qzblfgwbc0000gn/T/ipykernel_17191/1243002774.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Recap — Phase 2 complete

You went from a hand-weighted map to a **machine-learning flood-susceptibility model** (spatial-CV AUC ≈ 0.92), applied it across the city, and translated it into **people at risk by district**.

**Honest caveats** (same spirit as Phase 1): labels come from a single SAR-detected flood that can't see in-street urban waterlogging, so `built-up` partly encodes a sensor blind-spot; population at risk is an *exposure* estimate, not a casualty forecast. These are exactly the things a v3 (rainfall + assembled waterlogging labels) would address.